<a href="https://www.kaggle.com/code/shivamvachhnai/gru-lstm-sentiment-analysis-on-imdb-dataset?scriptVersionId=316391908" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [41]:
import pandas as pd 
import numpy as np
import torch 
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

In [42]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [43]:
df = pd.read_csv("/kaggle/input/datasets/waseemalastal/imdb-dataset/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [44]:
import re 
import string 
exclude = string.punctuation

def pre_process_data(text):
    
    #Lower text 
    text=str(text).lower().strip()
    
    #Remove Html tags
    pattern = re.compile('<.*?>')
    text = pattern.sub(r'',text)
    
    #Remove puctuation 
    text = text.translate(str.maketrans('','',exclude))

    #Sent tokenized document 
    return text.split(" ")

df["review"] = df["review"].apply(pre_process_data)

In [45]:
df["sentiment"] = df["sentiment"].apply(lambda x:0 if x == 'negative' else 1)
df.head()

,review,sentiment
0,"[one, of, the, other, reviewers, has, mentione...",1
1,"[a, wonderful, little, production, the, filmin...",1
2,"[i, thought, this, was, a, wonderful, way, to,...",1
3,"[basically, theres, a, family, where, a, littl...",0
4,"[petter, matteis, love, in, the, time, of, mon...",1


In [46]:
df["sentiment"].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

In [47]:
X = df.iloc[:,0]
y = df.iloc[:,1]

In [48]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,stratify=df['sentiment'])

In [49]:
y_train.value_counts()

sentiment
0    20000
1    20000
Name: count, dtype: int64

In [50]:
vocab = {'<PAD>':0,'<UNK>':1}

def build_vocab(row):
    for token in row:
        if token not in vocab:
            vocab[token] = len(vocab)
            
X_train.apply(build_vocab)

26231    None
15776    None
16743    None
29769    None
10506    None
         ... 
5565     None
16326    None
4594     None
28410    None
38714    None
Name: review, Length: 40000, dtype: object

In [51]:
len(vocab)
print(list(vocab.keys())[2:10])

['well', 'the', 'big', 'money', 'machine', 'has', 'done', 'it']


In [52]:
def text_to_indices(text,vocab):
    tokens = text if isinstance(text,list) else pre_process_data(text)
    return [vocab.get(token,vocab['<UNK>']) for token in tokens]

In [53]:
class MyCustomDataset(Dataset):
    
    def __init__(self,X,y,vocab):
        self.features = X
        self.label= y
        self.vocab = vocab
        
    def __len__(self):
        return self.features.shape[0]
        
    def __getitem__(self,index):
        numeric_doc = text_to_indices(self.features[index],self.vocab)
        numeric_label = self.label[index]
        
        return torch.tensor(numeric_doc),torch.tensor(numeric_label)

In [54]:
train_dataset = MyCustomDataset(X_train.values,y_train.values,vocab)
test_dataset = MyCustomDataset(X_test.values, y_test.values, vocab)

In [55]:
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    docs,labels = zip(*batch)
    docs_padded = pad_sequence(docs,batch_first=True,padding_value=0)
    labels = torch.tensor(labels,dtype=torch.long)
    return docs_padded,labels
    
train_dataloader = DataLoader(train_dataset,batch_size=32,collate_fn=collate_fn)
test_dataloader= DataLoader(test_dataset,batch_size=32,collate_fn=collate_fn)

In [56]:
class  ModelGRU(nn.Module):
    
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.GRU(50,64,batch_first=True)
        self.fc = nn.Linear(64,1)
        
    def forward(self,doc):
        embedded_doc = self.embedding(doc)
        output,hidden = self.rnn(embedded_doc)
        result = self.fc(hidden.squeeze(0))
        return result.squeeze(1)

In [57]:
class  ModelLSTM(nn.Module):
    
    def __init__(self,vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size,embedding_dim=50)
        self.rnn = nn.LSTM(50,64,batch_first=True)
        self.fc = nn.Linear(64,1)
        
    def forward(self,doc):
        embedded_doc = self.embedding(doc)
        itermidiate,(final_hidden,final_cell)= self.rnn(embedded_doc)
        result = self.fc(final_hidden.squeeze(0))
        return result.squeeze(1)

In [58]:
learning_rate=0.001
epochs=20

In [59]:
model_gru = ModelGRU(len(vocab))
model_LSTM = ModelLSTM(len(vocab))
model_gru.to(device)
model_LSTM.to(device)
optimizer_gru = torch.optim.Adam(model_gru.parameters(),lr=learning_rate)
optimizer_lstm = torch.optim.Adam(model_LSTM.parameters(),lr=learning_rate)
criterion = nn.BCEWithLogitsLoss()

In [60]:
for i in range(epochs):
    total_loss=0
    for batch_features,batch_labels in train_dataloader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        optimizer_gru.zero_grad()
        output = model_gru(batch_features)
        loss=criterion(output,batch_labels.float())
        loss.backward()
        optimizer_gru.step()
        total_loss += loss.item()
print(f"GRU Model FInal Loss:{total_loss/len(train_dataloader)}")

GRU Model FInal Loss:0.0022495794336231485


In [61]:
for i in range(epochs):
    total_loss=0
    for batch_features,batch_labels in train_dataloader:
        batch_features,batch_labels = batch_features.to(device),batch_labels.to(device)
        optimizer_lstm.zero_grad()
        output = model_LSTM(batch_features)
        loss=criterion(output,batch_labels.float())
        loss.backward()
        optimizer_lstm.step()
        total_loss += loss.item()
print(f"LSTM Model Final Loss:{total_loss/len(train_dataloader)}")

LSTM Model Final Loss:0.014204520930582658


In [62]:
model_gru.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_feature,batch_labels in test_dataloader:
        batch_feature,batch_labels = batch_feature.to(device),batch_labels.to(device)
        output = model_gru(batch_feature)
        prob = torch.sigmoid(output)
        pred = (prob >= 0.5).long()
        total += batch_labels.shape[0]
        correct += (pred == batch_labels).sum().item()
print(f"GRU Model Accuracy: {correct/total}")

GRU Model Accuracy: 0.8784


In [63]:
model_LSTM.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_feature,batch_labels in test_dataloader:
        batch_feature,batch_labels = batch_feature.to(device),batch_labels.to(device)
        output = model_LSTM(batch_feature)
        prob = torch.sigmoid(output)
        pred = (prob >= 0.5).long()
        total += batch_labels.shape[0]
        correct += (pred == batch_labels).sum().item()
print(f"LSTM Model Accuracy: {correct/total}")

LSTM Model Accuracy: 0.8618
